##AIによるカテゴリ分類の実装 ハンズオン（Section 11）

このノートブックでは、AIを使ってQAデータをカテゴリごとに分類し、ナレッジとして使えるデータと使えないデータに分ける流れを実行します。

-----

ここでやること

- カテゴリ分類によるデータ整理の仕組みを理解する  
- ナレッジとして使う／使わないの判断基準を理解する  
- プロンプトによって分類結果が変わる仕組みを理解する  
- 分類結果がナレッジの質に与える影響を理解する

In [ ]:
import requests
import zipfile
import shutil
import sqlite3

print("GithubからZIPファイルのダウンロードを開始します")

url = "https://github.com/aiknowledgelearning-decuments/document01/raw/main/section11/ank_base.zip"
r = requests.get(url)

with open("ank_base.zip", "wb") as f:
    f.write(r.content)

print("ank_base.zip のダウンロードが完了しました")

url = "https://github.com/aiknowledgelearning-decuments/document01/raw/main/section11/ank_tuned.zip"
r = requests.get(url)

with open("ank_tuned.zip", "wb") as f:
    f.write(r.content)

print("ank_tuned.zip のダウンロードが完了しました")

print("ZIPファイルを解凍します")

with zipfile.ZipFile("ank_base.zip", "r") as zip_ref:
    zip_ref.extractall(".")

with zipfile.ZipFile("ank_tuned.zip", "r") as zip_ref:
    zip_ref.extractall(".")

print("ZIPファイルの解凍が完了しました")
print("ank_base.db と ank_tuned.db を利用できる状態になりました")

print("ank_base.db を ank.db にコピーします")

shutil.copyfile("ank_base.db", "ank.db")

print("ank.db の作成が完了しました")

print("カテゴリ分類情報を初期化します")

conn = sqlite3.connect("ank.db")
cursor = conn.cursor()

cursor.execute("""
UPDATE qa
SET
    qa_category = NULL,
    qa_category_reason = NULL,
    is_knowledge_target = NULL
""")

conn.commit()
conn.close()

print("カテゴリ分類情報の初期化が完了しました")

In [ ]:
# Colabで config.py を作る（同じ作業ディレクトリに作成）
api_key = "XXXXXXXXXX"  # ←自分のキーに置き換え

with open("config.py", "w", encoding="utf-8") as f:
    f.write(f'OPENAI_API_KEY = "{api_key}"\n')

print("config.py を作成しました")

In [ ]:
# カテゴリ分類処理
import sqlite3
import json
import time
from openai import OpenAI
import config

DB_PATH = "ank.db"
MODEL_NAME = "gpt-4.1-mini"
CHUNK_SIZE = 100
client = OpenAI(api_key=config.OPENAI_API_KEY)

CATEGORY_LIST = ["policy", "definition", "fact", "context", "noise"]

SYSTEM_PROMPT = """
あなたはQAデータをナレッジ化するための分類担当です。
以下のQAを、必ず次の5カテゴリのいずれかに分類してください。
未分類、NULL、その他は禁止です。

カテゴリ定義:

noise:
挨拶、終了発言、中断、確認だけの発言、謝罪、感想、抽象的な意見など、
意味が薄くナレッジ化に向かないQA
context:
会議進行、発言者、出席者、手続き、質疑の順番、誰が何を確認したかなど、
その会議の文脈確認に関するQA
definition:
用語、仕組み、制度、法律、概念の説明に関するQA
fact:
具体的な事実、数値、時期、対象、場所、出来事、実施状況に関するQA
policy:
制度、政策、施策、方針、支援策、計画、政府の対応方針に関するQA

優先順位:
1. noise
2. context
3. definition
4. fact
5. policy

判定ルール:
- 挨拶、謝罪、確認だけ、意味が薄い発言は noise
- 会議の進行、発言者、出席者、順番、異議確認は context
- 用語や制度の意味を説明している場合は definition
- 数値、日時、場所、対象、出来事などの客観情報は fact
- 政策判断、制度変更、支援策、計画、政府方針は policy
- policy、definition、fact はナレッジ化対象
- context、noise はナレッジ化対象外
- 理由は短く、1文で書いてください
- 出力は必ずJSONのみ

出力形式:
{
  "category": "context",
  "is_knowledge_target": 0,
  "reason": "会議進行に関する内容のため"
}
"""

def classify_qa(question, answer):
    user_prompt = f"""
質問:
{question}

回答:
{answer}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0
    )

    content = response.choices[0].message.content.strip()
    return json.loads(content)

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

cursor.execute("""
SELECT COUNT(*)
FROM qa
WHERE qa_category IS NULL
   OR qa_category = ''
""")
remain_count = cursor.fetchone()[0]

print(f"未分類件数: {remain_count}")

if remain_count == 0:
    print("すべて分類済みです。")
    conn.close()
    raise SystemExit()

cursor.execute("""
SELECT qa_id, question, answer
FROM qa
WHERE qa_category IS NULL
   OR qa_category = ''
ORDER BY qa_id
LIMIT ?
""", (CHUNK_SIZE,))

rows = cursor.fetchall()
print(f"今回分類件数: {len(rows)}")

success_count = 0
error_count = 0

for idx, row in enumerate(rows, start=1):
    qa_id = row["qa_id"]
    question = row["question"] or ""
    answer = row["answer"] or ""

    try:
        result = classify_qa(question, answer)

        category = result.get("category")
        reason = result.get("reason", "")
        is_knowledge_target = int(result.get("is_knowledge_target", 0))

        if category not in CATEGORY_LIST:
            raise ValueError(f"不正なカテゴリです: {category}")

        if category in ["policy", "definition", "fact"]:
            is_knowledge_target = 1
        else:
            is_knowledge_target = 0

        cursor.execute("""
        UPDATE qa
           SET qa_category = ?,
               qa_category_reason = ?,
               is_knowledge_target = ?
         WHERE qa_id = ?
        """, (
            category,
            reason,
            is_knowledge_target,
            qa_id
        ))

        success_count += 1

        if idx % 20 == 0:
            conn.commit()
            print(f"{idx}件完了")

    except Exception as e:
        error_count += 1
        print(f"ERROR qa_id={qa_id}: {e}")

    time.sleep(0.1)

conn.commit()

print("=================================")
print("カテゴリ分類処理完了")
print(f"成功件数: {success_count}")
print(f"失敗件数: {error_count}")

cursor.execute("""
SELECT qa_category, is_knowledge_target, COUNT(*)
FROM qa
GROUP BY qa_category, is_knowledge_target
ORDER BY qa_category
""")

for row in cursor.fetchall():
    print(f"{row[0]} / target={row[1]} : {row[2]}件")

cursor.execute("""
SELECT COUNT(*)
FROM qa
WHERE qa_category IS NULL
   OR qa_category = ''
""")
remain_after = cursor.fetchone()[0]

print(f"残件数: {remain_after}")

if remain_after > 0:
    print("CHUNK_SIZE到達のため途中終了しました。再実行してください。")
else:
    print("全件分類が完了しました。")

conn.close()

In [ ]:
# カテゴリ分類結果の集計
import sqlite3

conn = sqlite3.connect("ank_tuned.db")
cursor = conn.cursor()

# カテゴリ別件数取得
cursor.execute("""
SELECT
    qa_category,
    COUNT(*) as cnt,
    is_knowledge_target
FROM qa
WHERE qa_category IS NOT NULL
GROUP BY qa_category, is_knowledge_target
ORDER BY is_knowledge_target, qa_category
""")

rows = cursor.fetchall()

# target別合計
target_totals = {}

for row in rows:
    category = row[0]
    count = row[1]
    target = row[2]

    if target not in target_totals:
        target_totals[target] = 0

    target_totals[target] += count

# 全体件数
total_count = sum(target_totals.values())

# ヘッダー
print(f"{'カテゴリ':<10} {'件数':>8} {'target':>8} {'target別合計':>8}")

displayed_targets = set()

# 明細表示
for row in rows:
    category = row[0]
    count = row[1]
    target = row[2]

    # target別合計は最初の1回だけ表示
    if target not in displayed_targets:
        target_total = target_totals[target]
        displayed_targets.add(target)
    else:
        target_total = ""

    print(f"{category:<15} {count:>8} {target:>8} {str(target_total):>12}")

# 合計表示
print("-" * 44)
print(f"{'合計':<12} {total_count:>8}")

# target比率
print("\n==============================")
print("target比率")

target_0 = target_totals.get(0, 0)
target_1 = target_totals.get(1, 0)

ratio_0 = (target_0 / total_count) * 100 if total_count > 0 else 0
ratio_1 = (target_1 / total_count) * 100 if total_count > 0 else 0

print(f"target=0 : {target_0:>8}件 ({ratio_0:.1f}%)")
print(f"target=1 : {target_1:>8}件 ({ratio_1:.1f}%)")

conn.close()

In [ ]:
# カテゴリー別のQAを3件づつ確認
import sqlite3
import pandas as pd

DB_FILE = "ank_tuned.db"

conn = sqlite3.connect(DB_FILE)

category_order = [
    "context",
    "noise",
    "definition",
    "fact",
    "policy"
]

for category in category_order:
    print("=" * 80)
    print(f"カテゴリ: {category}")
    print("=" * 80)

    df = pd.read_sql("""
    SELECT
        qa_id,
        question,
        answer,
        qa_category,
        qa_category_reason,
        is_knowledge_target
    FROM qa
    WHERE qa_category = ?
    ORDER BY qa_id
    LIMIT 3
    """, conn, params=(category,))

    if len(df) == 0:
        print("該当データはありません")
        print()
        continue

    for idx, row in df.iterrows():
        print(f"qa_id: {row['qa_id']}")
        print(f"質問: {row['question']}")
        print(f"回答: {row['answer']}")
        print(f"分類理由: {row['qa_category_reason']}")
        print(f"target: {row['is_knowledge_target']}")
        print("-" * 80)

    print()

conn.close()

In [ ]:
# 生成したQAをqaテーブルに保存
# tunedプロンプトを使う
# 対象DBは ank.db
# 実行前に qaテーブルの既存レコードを削除する

import sqlite3
import pandas as pd
from openai import OpenAI
import config
import json
import re

DB_FILE = "ank.db"
MODEL_NAME = "gpt-4.1-mini"
CHUNK_SIZE = 100   # 実行する発言件数

client = OpenAI(api_key=config.OPENAI_API_KEY)

conn = sqlite3.connect(DB_FILE)

print("qaテーブルの既存レコードを削除します")

conn.execute("DELETE FROM qa")

conn.commit()

print("qaテーブルの削除が完了しました")
print()

df_speeches = pd.read_sql("""
SELECT speechID, speaker, speech
FROM speeches
ORDER BY speechID
""", conn)

print("取得件数:", len(df_speeches))
print()

for i, (_, row) in enumerate(df_speeches.iterrows(), start=1):

    if i > CHUNK_SIZE:
        break

    speech_id = row["speechID"]
    speech_text = row["speech"]

    prompt = f"""
あなたは、国会議事録から検索に使えるQAを作成する編集者です。

以下の発言内容を読み取り、
ナレッジとして活用しやすい質問と回答を1〜5件作成してください。

重要:
- 必ず5件作る必要はありません。
- 内容が少ない場合は、1件または0件でもかまいません。
- 無理にQAを作らないでください。

質問作成ルール:
- 制度、政策、施策、方針、支援策、計画に関する質問を優先する
- 用語、制度、仕組み、背景、目的を理解するための質問を優先する
- 課題、懸念、対応方針、改善点が分かる質問を優先する
- 単なる数値、日付、件数、場所、過去の出来事だけを聞く質問はできるだけ避ける
- 会議進行、挨拶、謝辞、確認だけの内容はQAにしない
- 発言にない内容は追加しない
- 同じ意味の質問を複数作らない

回答作成ルール:
- 回答は発言本文に沿って書く
- 推測しない
- 簡潔に1〜3文でまとめる
- 数値や日時は、制度や方針の説明に必要な場合だけ使う

出力は必ずJSON配列のみとしてください。
説明文、前置き、補足は禁止です。

出力形式:
[
  {{
    "question": "質問文",
    "answer": "回答文"
  }}
]

発言内容:
{speech_text}
"""

    print("=" * 80)
    print(f"{i}件目 / speechID = {speech_id}")
    print(f"発言者: {row['speaker']}")
    print("-" * 80)

    try:

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )

        answer = response.choices[0].message.content.strip()

        answer = re.sub(r"^```json\s*", "", answer.strip())
        answer = re.sub(r"\s*```$", "", answer.strip())

        qa_list = json.loads(answer)

    except Exception as e:
        print("JSON変換エラー")
        print(e)
        continue

    for qa in qa_list:

        question = qa.get("question", "").strip()
        answer_text = qa.get("answer", "").strip()

        if not question or not answer_text:
            continue

        conn.execute("""
        INSERT INTO qa (speechID, question, answer)
        VALUES (?, ?, ?)
        """, (speech_id, question, answer_text))

    conn.commit()

    print(f"qaテーブルへ {len(qa_list)} 件登録しました")
    print()

conn.close()

print("処理完了")